# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [8]:
# your code here
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv("tic-tac-toe.csv")  


label_encoder = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':  
        df[col] = label_encoder.fit_transform(df[col])


X = df.drop(columns=["class"])  
y = df["class"]  


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [9]:
# your code here
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)), 
    Dense(8, activation='relu'),
    Dense(2, activation='softmax')  
])


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test))


model.save("tic-tac-toe.model")

Epoch 1/50
96/96 [==============================] - 1s 5ms/step - loss: 0.6354 - accuracy: 0.6554 - val_loss: 0.6194 - val_accuracy: 0.6719
Epoch 2/50
96/96 [==============================] - 0s 2ms/step - loss: 0.6182 - accuracy: 0.6645 - val_loss: 0.6106 - val_accuracy: 0.6667
Epoch 3/50
96/96 [==============================] - 0s 2ms/step - loss: 0.6068 - accuracy: 0.6684 - val_loss: 0.6032 - val_accuracy: 0.6771
Epoch 4/50
96/96 [==============================] - 0s 2ms/step - loss: 0.5966 - accuracy: 0.6658 - val_loss: 0.5940 - val_accuracy: 0.6875
Epoch 5/50
96/96 [==============================] - 0s 2ms/step - loss: 0.5869 - accuracy: 0.6880 - val_loss: 0.5884 - val_accuracy: 0.6875
Epoch 6/50
96/96 [==============================] - 0s 2ms/step - loss: 0.5768 - accuracy: 0.6919 - val_loss: 0.5808 - val_accuracy: 0.6927
Epoch 7/50
96/96 [==============================] - 0s 2ms/step - loss: 0.5657 - accuracy: 0.7023 - val_loss: 0.5717 - val_accuracy: 0.7031
Epoch 8/50
96/96 [==

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [10]:
# your code here

model = tf.keras.models.load_model("tic-tac-toe.model")


indices = np.random.choice(len(X_test), 5, replace=False)
X_sample = X_test[indices]
y_sample = y_test.iloc[indices]  


predictions = model.predict(X_sample)


for i in range(len(X_sample)):
    print(f"Actual: {y_sample.iloc[i]}, Predicted: {np.argmax(predictions[i])}")

1/1 [==============================] - 0s 108ms/step
Actual: False, Predicted: 0
Actual: True, Predicted: 1
Actual: False, Predicted: 1
Actual: True, Predicted: 1
Actual: True, Predicted: 1


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [26]:
# your code here
import tensorflow as tf
from tensorflow.keras import layers, models

# بناء النموذج
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

# تحميل البيانات
df = pd.read_csv("tic-tac-toe.csv")

# تحويل البيانات النصية إلى رقمية باستخدام LabelEncoder
label_encoder = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = label_encoder.fit_transform(df[col])

# فصل البيانات إلى متغيرات مستقلة (X) ومتغير تابع (y)
X = df.drop(columns=["class"])
y = df["class"]

# تحويل التسميات إلى ترميز one-hot encoding
y = to_categorical(y, num_classes=3)  # افترض أن لديك 3 فئات

# تقسيم البيانات إلى مجموعات تدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# تطبيق StandardScaler على البيانات
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# بناء النموذج
def build_model(input_shape):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_shape,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(3, activation='softmax')  # 3 فئات
    ])
    return model

# ضبط معدل التعلم
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# تجميع النموذج
input_shape = X_train.shape[1]  # الحصول على عدد الميزات في بيانات التدريب
model = build_model(input_shape)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# تدريب النموذج
history = model.fit(
    X_train, y_train,
    epochs=50,  # يمكنك زيادة عدد العهود
    batch_size=32,
    validation_data=(X_test, y_test)
)

# تقييم النموذج
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss: {loss}, Accuracy: {accuracy}')

# حفظ النموذج
model.save('improved_model')

# تحميل النموذج المحفوظ
loaded_model = tf.keras.models.load_model('improved_model')

# عمل تنبؤات
predictions = loaded_model.predict(X_test)
print(predictions)

Epoch 1/50
24/24 [==============================] - 1s 14ms/step - loss: 0.8161 - accuracy: 0.6462 - val_loss: 0.7109 - val_accuracy: 0.6510
Epoch 2/50
24/24 [==============================] - 0s 4ms/step - loss: 0.6405 - accuracy: 0.6828 - val_loss: 0.6287 - val_accuracy: 0.6875
Epoch 3/50
24/24 [==============================] - 0s 3ms/step - loss: 0.5802 - accuracy: 0.7232 - val_loss: 0.5944 - val_accuracy: 0.7083
Epoch 4/50
24/24 [==============================] - 0s 4ms/step - loss: 0.5416 - accuracy: 0.7520 - val_loss: 0.5695 - val_accuracy: 0.7240
Epoch 5/50
24/24 [==============================] - 0s 3ms/step - loss: 0.5101 - accuracy: 0.7728 - val_loss: 0.5455 - val_accuracy: 0.7396
Epoch 6/50
24/24 [==============================] - 0s 4ms/step - loss: 0.4774 - accuracy: 0.7859 - val_loss: 0.5197 - val_accuracy: 0.7552
Epoch 7/50
24/24 [==============================] - 0s 3ms/step - loss: 0.4516 - accuracy: 0.7963 - val_loss: 0.4950 - val_accuracy: 0.7656
Epoch 8/50
24/24 [=

**Which approach(es) did you find helpful to improve your model performance?**

In [14]:
# your answer here
I found that adjusting the learning rate using the Adam optimizer and adding additional layers helped improve my model's performance.  
Additionally, increasing the number of epochs allowed the model to learn better, but I had to be careful to avoid overfitting.  
Dropout layers also helped in preventing overfitting, which led to better generalization on the test dataset.